# Sales Data Cleaning and Analysis

This notebook provides a professional end-to-end workflow for cleaning raw sales data, computing business KPIs, and building visualizations with Matplotlib and Plotly.

In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from pathlib import Path

plt.style.use('seaborn-v0_8')

## Load Raw Dataset

Read the raw sales data from the `data/` folder and inspect the dataset structure.

In [29]:
data_path = Path('..') / 'data' / 'Sample - Superstore_Orders.csv'
raw_df = pd.read_csv(data_path, dtype=str)
raw_df.head()

,Category,City,Country/Region,Customer ID,Customer Name,Order Date,Order ID,Postal Code,Product ID,Product Name,...,Ship Status,State,Sub-Category,Days to Ship Actual,Days to Ship Scheduled,Discount,Profit,Quantity,Sales,Sales Forecast
0,Furniture,Henderson,United States,CG-12520,Claire Gute,11/8/2019,CA-2019-152156,42420,FUR-BO-10001798,Bush Somerset Collection Bookcase,...,Shipped On Time,Kentucky,Bookcases,3,3,0.00%,$42,2,$262,$392
1,Furniture,Henderson,United States,CG-12520,Claire Gute,11/8/2019,CA-2019-152156,42420,FUR-CH-10000454,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",...,Shipped On Time,Kentucky,Chairs,3,3,0.00%,$220,3,$732,"$1,096"
2,Office Supplies,Los Angeles,United States,DV-13045,Darrin Van Huff,6/12/2019,CA-2019-138688,90036,OFF-LA-10000240,Self-Adhesive Address Labels for Typewriters b...,...,Shipped Late,California,Labels,4,3,0.00%,$7,2,$15,$22
3,Furniture,Fort Lauderdale,United States,SO-20335,Sean O'Donnell,10/11/2018,US-2018-108966,33311,FUR-TA-10000577,Bretford CR4500 Series Slim Rectangular Table,...,Shipped Late,Florida,Tables,7,6,45.00%,($383),5,$958,"$1,434"
4,Office Supplies,Fort Lauderdale,United States,SO-20335,Sean O'Donnell,10/11/2018,US-2018-108966,33311,OFF-ST-10000760,Eldon Fold 'N Roll Cart System,...,Shipped Late,Florida,Storage,7,6,20.00%,$3,2,$22,$33


In [30]:
raw_df.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 25 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   Category                9994 non-null   str  
 1   City                    9994 non-null   str  
 2   Country/Region          9994 non-null   str  
 3   Customer ID             9994 non-null   str  
 4   Customer Name           9994 non-null   str  
 5   Order Date              9994 non-null   str  
 6   Order ID                9994 non-null   str  
 7   Postal Code             9983 non-null   str  
 8   Product ID              9994 non-null   str  
 9   Product Name            9994 non-null   str  
 10  Region                  9994 non-null   str  
 11  Row ID                  9994 non-null   str  
 12  Segment                 9994 non-null   str  
 13  Ship Date               9994 non-null   str  
 14  Ship Mode               9994 non-null   str  
 15  Ship Status             9994 non

## Cleaning Helpers

Define reusable functions to normalize numeric fields and make the dataset analysis-ready.

In [31]:
def clean_numeric_series(series):
    series = series.astype(str).str.strip()
    series = series.str.replace(r'[\$,]', '', regex=True)
    series = series.str.replace(r'\(', '-', regex=True)
    series = series.str.replace(r'\)', '', regex=True)
    series = series.str.replace('%', '', regex=False)
    series = series.str.replace(r'\s+', '', regex=True)
    return pd.to_numeric(series, errors='coerce')

def clean_sales_data(df):
    df = df.copy()
    df.columns = df.columns.str.strip()
    df = df.drop_duplicates(ignore_index=True)

    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype(str).str.strip()

    if 'Postal Code' in df.columns:
        df['Postal Code'] = df['Postal Code'].replace({'nan': pd.NA}).fillna('Unknown')

    for col in ['Sales', 'Profit', 'Discount', 'Sales Forecast']:
        if col in df.columns:
            df[col] = clean_numeric_series(df[col])

    for date_col in ['Order Date', 'Ship Date']:
        if date_col in df.columns:
            df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

    required = [col for col in ['Order ID', 'Order Date', 'Product ID', 'Sales', 'Profit'] if col in df.columns]
    df = df.dropna(subset=required).reset_index(drop=True)

    if 'Order Date' in df.columns:
        df['Order Year'] = df['Order Date'].dt.year
        df['Order Month'] = df['Order Date'].dt.month

    if {'Profit', 'Sales'}.issubset(df.columns):
        df['Profit Margin'] = df.apply(lambda row: row['Profit'] / row['Sales'] if row['Sales'] != 0 else pd.NA, axis=1)

    return df

## Clean the Dataset

Execute the cleaning pipeline and validate the corrected types and feature engineering.

In [32]:
df = clean_sales_data(raw_df)
df.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_28228\1632984138.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:


,Category,City,Country/Region,Customer ID,Customer Name,Order Date,Order ID,Postal Code,Product ID,Product Name,...,Days to Ship Actual,Days to Ship Scheduled,Discount,Profit,Quantity,Sales,Sales Forecast,Order Year,Order Month,Profit Margin
0,Furniture,Henderson,United States,CG-12520,Claire Gute,2019-11-08,CA-2019-152156,42420,FUR-BO-10001798,Bush Somerset Collection Bookcase,...,3,3,0.0,42,2,262,392,2019,11,0.160305
1,Furniture,Henderson,United States,CG-12520,Claire Gute,2019-11-08,CA-2019-152156,42420,FUR-CH-10000454,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",...,3,3,0.0,220,3,732,1096,2019,11,0.300546
2,Office Supplies,Los Angeles,United States,DV-13045,Darrin Van Huff,2019-06-12,CA-2019-138688,90036,OFF-LA-10000240,Self-Adhesive Address Labels for Typewriters b...,...,4,3,0.0,7,2,15,22,2019,6,0.466667
3,Furniture,Fort Lauderdale,United States,SO-20335,Sean O'Donnell,2018-10-11,US-2018-108966,33311,FUR-TA-10000577,Bretford CR4500 Series Slim Rectangular Table,...,7,6,45.0,-383,5,958,1434,2018,10,-0.399791
4,Office Supplies,Fort Lauderdale,United States,SO-20335,Sean O'Donnell,2018-10-11,US-2018-108966,33311,OFF-ST-10000760,Eldon Fold 'N Roll Cart System,...,7,6,20.0,3,2,22,33,2018,10,0.136364


In [33]:
df.dtypes

Category                             str
City                                 str
Country/Region                       str
Customer ID                          str
Customer Name                        str
Order Date                datetime64[us]
Order ID                             str
Postal Code                          str
Product ID                           str
Product Name                         str
Region                               str
Row ID                               str
Segment                              str
Ship Date                 datetime64[us]
Ship Mode                            str
Ship Status                          str
State                                str
Sub-Category                         str
Days to Ship Actual                  str
Days to Ship Scheduled               str
Discount                         float64
Profit                             int64
Quantity                             str
Sales                              int64
Sales Forecast  

In [34]:
missing_summary = df.isna().sum().sort_values(ascending=False)
missing_summary[missing_summary > 0]

Profit Margin    1
dtype: int64

## Key Performance Indicators

Calculate the primary KPIs for sales performance and order activity.

In [35]:
total_sales = df['Sales'].sum()
total_profit = df['Profit'].sum()
average_order_value = df.groupby('Order ID')['Sales'].sum().mean()
top_products = df.groupby('Product Name', as_index=False)['Sales'].sum().nlargest(10, 'Sales')
top_categories = df.groupby('Category', as_index=False)['Sales'].sum().sort_values('Sales', ascending=False)
monthly_sales = df.groupby(['Order Year', 'Order Month'], as_index=False)['Sales'].sum().sort_values(['Order Year', 'Order Month'])
total_sales, total_profit, average_order_value

(np.int64(2297354), np.int64(286347), np.float64(458.645238570573))

In [36]:
print(f'Total Sales: ${total_sales:,.2f}')
print(f'Total Profit: ${total_profit:,.2f}')
print(f'Average Order Value: ${average_order_value:,.2f}')

Total Sales: $2,297,354.00
Total Profit: $286,347.00
Average Order Value: $458.65


## Top Products and Categories

Visualize the best-selling products and categories.

In [37]:
fig1 = px.bar(top_products, x='Sales', y='Product Name', orientation='h', title='Top 10 Products by Sales', template='plotly_white')
fig1.update_layout(yaxis={'categoryorder': 'total ascending'})
fig1.show()

In [38]:
fig2 = px.pie(top_categories.head(6), names='Category', values='Sales', title='Sales by Category', hole=0.4, template='plotly_white')
fig2.show()

## Trend and Regional Analysis

Analyze monthly sales trends and profit performance by region.

In [42]:
monthly_sales['Period'] = pd.to_datetime({
    'year': monthly_sales['Order Year'],
    'month': monthly_sales['Order Month'],
    'day': 1
})
fig3 = px.line(monthly_sales, x='Period', y='Sales', markers=True, title='Monthly Sales Trend', template='plotly_white')
fig3.show()

In [43]:
region_profit = df.groupby('Region', as_index=False)['Profit'].sum().sort_values('Profit', ascending=False)
fig4 = px.bar(region_profit, x='Region', y='Profit', title='Profit by Region', template='plotly_white')
fig4.show()

## Save the Cleaned Dataset

Export the cleaned dataset to `output/cleaned_sales.csv` for later analysis and dashboard generation.

In [44]:
output_path = Path('..') / 'output' / 'cleaned_sales.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)
print(f'Cleaned data saved to {output_path}')

Cleaned data saved to ..\output\cleaned_sales.csv
